<a href="https://colab.research.google.com/github/abdhmohammadi/NLP/blob/main/Underestanding-Core-of-LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Underestanding core of Large Language Models
**Date:** 2025-07-15

**Abdh. Mohammdi**

#Introduction
The **core mathematical optimization problem** in **Natural Language Processing (NLP)** — and especially in **Large Language Models (LLMs)** — is to maximize the Likelihood of Text data, Given a large corpus $\mathcal{D} = \{x_1, x_2, ..., x_N\}$, where each $x_i = (w_1, w_2, ..., w_T)$ is a sequence of words or tokens $w_i$, a NLP model maximizes the log-likelihood of the data under model $P_\theta(x)$

$$
\boxed{
\theta^* = \arg\max_\theta \sum_{i=1}^N \log P_\theta(x_i) = \arg\max_\theta \sum_{i=1}^N\log P_\theta(w_1,w_2,…, w_T)
}
$$

In practice, for autoregressive language models (e.g., GPT), this is decomposed into **next-token prediction**: $$
\log P_\theta(w_1, w_2, \dots, w_T) = \sum_{t=1}^T \log P_\theta(w_t \mid w_1, w_2, ..., w_{t-1})
$$
and **model parameters** $\theta$ (weights of a neural network) are optimized using **gradient descent** on the **negative log-likelihood (cross-entropy loss)**.

$$
\mathcal{L}(\theta) = - \sum_{i=1}^N \sum_{t=1}^{T_i} \log P_\theta(w_t^{(i)} \mid w_1^{(i)}, ..., w_{t-1}^{(i)})
$$

This is equivalent to minimizing **cross-entropy** between the predicted distribution and the true next word.(**[1]-Goodfellow et al., 2016**).


#$P(w_1)$: Probability of the first token
To generate a sentence $x_i = (w_1, \dots, w_T)$, the model computes the joint probability $P(x_i)$ using the **chain rule** of probability:
$$P( x_i) = \prod_{t=1}^{T} P(w_t \mid w_1, \dots, w_{t-1}).$$

which is an autoregressive factorization capturing all prior context. Unlike **n-gram** models that rely on the **Markov assumption**, large language models (e.g., GPT) use **attention mechanisms** to capture dependencies over the entire preceding context.(**[2]-Vaswani et al., 2017; [3]-Radford et al., 2018**).

In this note, to understand these models more deeply, we first calculate the probability of $P(w_1)$, the first token of a sentence using the traditional method. Then ...

#Corpus
We define a corpus of 20 sentences as our 'training vocabulary'.
For demonstration, these are manually selected general-purpose sentences.

In [ ]:
corpus =[
    "The quick brown fox jumps over the lazy dog",
    "She sells seashells by the seashore",
    "All that glitters is not gold",
    "To be or not to be that is the question",
    "I think therefore I am",
    "The only thing we have to fear is fear itself",
    "That's one small step for man one giant leap for mankind",
    "In the beginning God created the heaven and the earth",
    "It was the best of times it was the worst of times",
    "Call me Ishmael",
    "It is a truth universally acknowledged that a single man in possession of a good fortune must be in want of a wife",
    "May the Force be with you",
    "Elementary my dear Watson",
    "I have a dream that one day this nation will rise up",
    "Four score and seven years ago our fathers brought forth on this continent a new nation",
    "Ask not what your country can do for you ask what you can do for your country",
    "The only limit to our realization of tomorrow is our doubts of today",
    "Float like a butterfly sting like a bee",
    "The pen is mightier than the sword",
    "Knowledge is power"
]

#Compute $P(w_1)$ for All Sentences



The code computes the probability of the first word of a sentence occurring in the corpus.
Let $\mathcal{D} = \{x_1, x_2, \dots, x_{20}\}$ be the corpus of $20$ sentences.
Let $x_i = (w_{i,1}, w_{i,2}, \dots, w_{i, L_i})$ be the sequence of tokens in sentence $x_i$, where $L_i$ is the length of sentence $x_i$.

The code calculates the probability of each unique first token $w_1$ appearing in the corpus.
The set of all first tokens is $W_1 = \{w_{1,1}, w_{2,1}, \dots, w_{20,1}\}$.
The probability of a specific token $w$ being the first token is given by the empirical probability:

$P(w_1 = w) = \frac{\text{Number of sentences in } \mathcal{D}\text{ that start with token } w}{N}$

This is computed in the below code cell.

In [ ]:
from collections import Counter
import pandas as pd

# Tokenizing each sentence and extracting first tokens
tokenized = [s.split() for s in corpus]
start_tokens = [tokens[0] for tokens in tokenized]

# Frequency counts
counts = Counter(start_tokens)
total = len(start_tokens)

# Prepare rows
first_words = list(counts.keys())
frequency_row = [counts[w] for w in first_words]
fraction_row = [f"{counts[w]}/{total}" for w in first_words]
probability_row = [round(counts[w] / total, 6) for w in first_words]

# Build final DataFrame
data = {
    "First Word": ["Frequency", "P(wi)", "Probability"]
}
for word in first_words:
    data[word] = [ counts[word], f"{counts[word]}/{total}", round(counts[word] / total, 6) ]

df = pd.DataFrame(data)

df.head()


,First Word,The,She,All,To,I,That's,In,It,Call,May,Elementary,Four,Ask,Float,Knowledge
0,Frequency,4,1,1,1,2,1,1,2,1,1,1,1,1,1,1
1,P(wi),4/20,1/20,1/20,1/20,2/20,1/20,1/20,2/20,1/20,1/20,1/20,1/20,1/20,1/20,1/20
2,Probability,0.2,0.05,0.05,0.05,0.1,0.05,0.05,0.1,0.05,0.05,0.05,0.05,0.05,0.05,0.05


The output shows frequency of start words for each sentence. Because the number of terms is 20, all of these values are divided by 20, which is in the second row, the third row shows the result of the calculation as a decimal number. for example $P('the')=\frac{4}{20}=0.2, P('I')=\frac{2}{20}=0.1$. for other words this is $\frac{1}{20}=0.05$.

# Joint Probability
Based of this approach we want to compute $P(\text{"She sells"})$.

$P(\text{"She"}) = \frac{\text{sentences starting with "she"}}{\text{total sentences}} = \frac{1}{20} = 0.05$.

Just one sentence exist starting with "She sells" in the corpus, therfore:
$P("sells"|"She")=1.0$

so $P(\text{"She sells"})=P("She").P("sells"|"She")=0.05\times 1.0 = 0.05$

To generate **"She can sells the pen."** we want to compute the **joint probability** of this sentence based on the corpus using:

$$
P(w_1, w_2, \dots, w_n) = P(w_1) \cdot P(w_2 \mid w_1) \cdot P(w_3 \mid w_2) \cdot \cdots \cdot P(w_n \mid w_{n-1})
$$

This assumes a **bigram model**, i.e., we use **only 1 previous word** for each conditional probability. Let’s tokenize and lowercase:

```python
["she", "can", "sells", "the", "pen"]
```
We computed $P("She")= 0.05$. "can" never follows "she" and $P(\text{"can"} \mid \text{"she"}) = 0$. This kills the entire joint probability:

$$
P(\text{"she can sells the pen"}) = 0.05 \times 0 = 0
$$
So: **Joint probability = 0** — This sentence is **not supported by our corpus**. We can fix this useing Smoothing (e.g., Laplace). This assigns small probability to **unseen bigrams**.

$$
P(w_i \mid w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i) + 1}{\text{count}(w_{i-1}) + V}
$$

Where $V$ is the vocabulary size.

## 5. Generating Logits $z_i$ in Neural Network Methods
In neural networks used for language modeling (like Transformers), the model doesn't directly output probabilities like $P(w_i)$. Instead, it outputs logits, usually denoted $z_i$. Logits are not probabilities. they can be any real number, even negative, These are raw, unnormalized scores. Neural networks compute logits via: $z = W h_{t-1} + b$.

To get the actual probability $P(w_i)$ of choosing token $w_i$, It is applied the softmax function:
$P(w_i)=\frac{e^{z_i}}{\sum_{i=1}^N e^{z_j}}$

## Compute Logits $z_i$ for Selected Sentence using GPT-2

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

sentence = "She can sells the pen."

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

# Tokenize sentence with tokenizer (include special tokens by default)
inputs = tokenizer(sentence, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits  # shape: (1, seq_len, vocab_size)
logits = logits[0]      # remove batch dimension -> (seq_len, vocab_size)

# Get token IDs of the input sentence (including special tokens)
input_ids = inputs['input_ids'][0]  # shape: (seq_len,)

# Extract logits for each token predicted at previous position
# For token i, the logits at position i-1 predict token i
# So, for token i in input_ids[1:], logits[i-1] corresponds to that prediction

z_values = []

for i in range(1, len(input_ids)):
    token_id = input_ids[i].item()
    logit_score = logits[i - 1, token_id].item()
    z_values.append(logit_score)

# For the first token, no previous token to predict it, so can set None or ignore
z_values = [None] + z_values

print("Tokens:", tokenizer.convert_ids_to_tokens(input_ids))
print("Logits", z_values[1:])


Tokens: ['She', 'Ġcan', 'Ġsells', 'Ġthe', 'Ġpen', '.']
Logits [-36.10849380493164, -133.4515838623047, -98.04438781738281, -95.28128051757812, -114.34510040283203]


## Compute probability for lagits $z_i$:

In [ ]:
import torch.nn.functional as F

for i in range(1, len(input_ids)):
    logits_vector = logits[i-1]  # vector of shape [vocab_size]
    probs = F.softmax(logits_vector, dim=0)  # softmax over vocab dimension
    token_id = input_ids[i].item()
    prob_token = probs[token_id].item()
    print(f"Probability of token {tokenizer.convert_ids_to_tokens([token_id])[0]}: {prob_token:0.6f}")



Probability of token Ġcan: 0.006951
Probability of token Ġsells: 0.000000
Probability of token Ġthe: 0.053094
Probability of token Ġpen: 0.000072
Probability of token .: 0.032682


## Compute probabilitiy for $z_i$ manually:

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import math

# Load model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()

sentence = "She can sells the pen."
inputs = tokenizer(sentence, return_tensors='pt')

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits[0]  # shape: (seq_len, vocab_size)
input_ids = inputs['input_ids'][0]  # shape: (seq_len,)

# Manual softmax function
def manual_softmax(logits_vector):
    max_logit = torch.max(logits_vector)   # for numerical stability
    exp_logits = torch.exp(logits_vector - max_logit)  # subtract max to avoid large exponentials
    sum_exp = torch.sum(exp_logits)
    return exp_logits / sum_exp

# Compute probabilities of actual tokens step-by-step
prob_values = [None]  # first token no prediction (no previous token)

for i in range(1, len(input_ids)):
    logits_vector = logits[i -1]  # logits predicting token i
    probs = manual_softmax(logits_vector)
    token_id = input_ids[i].item()
    prob_token = probs[token_id].item()
    log_prob_token = math.log(prob_token)
    print(f"Probability of token {tokenizer.convert_ids_to_tokens([token_id])[0]}: {prob_token:.6f}")#, Log-Probability: {log_prob_token:.6f}")
    prob_values.append(prob_token)


Probability of token Ġcan: 0.006951
Probability of token Ġsells: 0.000000
Probability of token Ġthe: 0.053094
Probability of token Ġpen: 0.000072
Probability of token .: 0.032681


## 10. Conclusion
We contrasted traditional and neural methods, showing chain probabilities and logits conversion via softmax.

## 11. References
[1] - Goodfellow, I., Bengio, Y., & Courville, A. (2016). Deep Learning. MIT Press.
https://www.deeplearningbook.org

[2] - Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). *Attention Is All You Need*. Advances in Neural Information Processing Systems (NeurIPS). [https://arxiv.org/abs/1706.03762](https://arxiv.org/abs/1706.03762)

[3] - Radford, A., Narasimhan, K., Salimans, T., & Sutskever, I. (2018). *Improving Language Understanding by Generative Pre-Training*. OpenAI Technical Report. [https://www.cs.ubc.ca/~amuham01/LING530/papers/radford2018improving.pdf](https://www.cs.ubc.ca/~amuham01/LING530/papers/radford2018improving.pdf)

- Papoulis & Pillai, *Probability, Random Variables and Stochastic Processes*, 2002